# SPARQL Embeddings Demo

This notebook trains and evaluates a **Hybrid SPARQL encoder** on semantic triplets,
and compares its embedding space against the **Qwen3-Embedding** baseline.

Sections:
1. Setup and imports
2. Configuration and reproducibility
3. Data loading and split handling
4. Dataset & DataLoaders
5. Model loading and training


In [1]:
import sys, os
if "src" not in sys.path:
    sys.path.append(os.path.abspath("src"))

import json, random, numpy as np, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from src.config import Config
from src.graph_builder import SPARQLGraphBuilder
from src.model import HybridSPARQLEncoder
from src.inference import SPARQLEmbedder
from src.semantic_dataset import SemanticSPARQLDataset, semantic_collate_fn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SEMANTIC_PATH = "dataset/semantic_triplets_dbpedia.json"

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

print(f"Total semantic entries: {len(raw_sem)}")

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/ganesh-jupyterhub/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total semantic entries: 3858


In [2]:
indices = list(range(len(raw_sem)))
random.shuffle(indices)

n = len(indices)
train_size = int(0.8 * n)
valid_size = int(0.1 * n)
test_size  = n - train_size - valid_size

train_idx = indices[:train_size]
valid_idx = indices[train_size : train_size + valid_size]
test_idx  = indices[train_size + valid_size :]

print(f"Train: {len(train_idx)}, Valid: {len(valid_idx)}, Test: {len(test_idx)}")

os.makedirs("dataset/splits", exist_ok=True)
split_info = {
    "seed": SEED,
    "train_idx": train_idx,
    "valid_idx": valid_idx,
    "test_idx": test_idx,
}
with open("dataset/splits/semantic_triplet_splits_dbpedia.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=2)

print("Saved splits to dataset/splits/semantic_triplet_splits_dbpedia.json")


Train: 3086, Valid: 385, Test: 387
Saved splits to dataset/splits/semantic_triplet_splits_dbpedia.json


In [3]:
print("Initializing semantic dataset...")
builder = SPARQLGraphBuilder()
full_sem = SemanticSPARQLDataset(SEMANTIC_PATH, builder)
print(f"Total semantic triplets: {len(full_sem)}")

train_ds = Subset(full_sem, train_idx)
valid_ds = Subset(full_sem, valid_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=Config.BATCH_SIZE,
    collate_fn=semantic_collate_fn,
    shuffle=True,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=Config.BATCH_SIZE,
    collate_fn=semantic_collate_fn,
    shuffle=False,
)

print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)}")


Initializing semantic dataset...
Loading semantic dataset from dataset/semantic_triplets_dbpedia.json...
Generated 37979 semantic training triplets.
Total semantic triplets: 37979
Train: 3086 | Valid: 385


In [4]:
def infonce_inbatch(anchor_emb, pos_emb, temperature: float):
    anchor_emb = F.normalize(anchor_emb, dim=1)
    pos_emb = F.normalize(pos_emb, dim=1)
    logits = (anchor_emb @ pos_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


In [5]:
model = HybridSPARQLEncoder().to(Config.DEVICE)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

print(f" Starting training on {Config.DEVICE} with {len(train_ds)} semantic samples...")

for epoch in range(Config.EPOCHS):
    total_loss = 0.0

    for anchor, pos, neg in train_loader:
        anchor = anchor.to(Config.DEVICE)
        pos = pos.to(Config.DEVICE)

        optimizer.zero_grad()
        a_emb = model(anchor)
        p_emb = model(pos)
        loss = infonce_inbatch(a_emb, p_emb, Config.TEMPERATURE)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{Config.EPOCHS} | Train Loss: {avg_train_loss:.4f}")

    # simple validation sim check
    model.eval()
    with torch.no_grad():
        sims_pos, sims_neg, count = 0.0, 0.0, 0
        for anchor, pos, neg in valid_loader:
            anchor = anchor.to(Config.DEVICE)
            pos = pos.to(Config.DEVICE)
            neg = neg.to(Config.DEVICE)

            a_emb = F.normalize(model(anchor), dim=1)
            p_emb = F.normalize(model(pos), dim=1)
            n_emb = F.normalize(model(neg), dim=1)

            sims_pos += torch.sum(torch.sum(a_emb * p_emb, dim=1)).item()
            sims_neg += torch.sum(torch.sum(a_emb * n_emb, dim=1)).item()
            count += anchor.size(0)

        print(
            f"          Valid sim(anchor,pos): {sims_pos / count:.4f} | "
            f"sim(anchor,neg): {sims_neg / count:.4f}"
        )

    model.train()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 395.73it/s, Materializing param=pooler.dense.weight]                               


 Starting training on cuda with 3086 semantic samples...
Epoch 1/20 | Train Loss: 2.0979
          Valid sim(anchor,pos): 0.3042 | sim(anchor,neg): 0.1493
Epoch 2/20 | Train Loss: 1.5973
          Valid sim(anchor,pos): 0.3017 | sim(anchor,neg): 0.1094
Epoch 3/20 | Train Loss: 1.4972
          Valid sim(anchor,pos): 0.2994 | sim(anchor,neg): 0.1152
Epoch 4/20 | Train Loss: 1.4177
          Valid sim(anchor,pos): 0.2975 | sim(anchor,neg): 0.0845
Epoch 5/20 | Train Loss: 1.3460
          Valid sim(anchor,pos): 0.2947 | sim(anchor,neg): 0.1029
Epoch 6/20 | Train Loss: 1.3023
          Valid sim(anchor,pos): 0.2925 | sim(anchor,neg): 0.0977
Epoch 7/20 | Train Loss: 1.2148
          Valid sim(anchor,pos): 0.2919 | sim(anchor,neg): 0.0869
Epoch 8/20 | Train Loss: 1.1119
          Valid sim(anchor,pos): 0.2825 | sim(anchor,neg): 0.0547
Epoch 9/20 | Train Loss: 1.0182
          Valid sim(anchor,pos): 0.2838 | sim(anchor,neg): 0.0661
Epoch 10/20 | Train Loss: 0.9281
          Valid sim(anchor,p

In [6]:
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/final_model_semantic-20.pt")
print("Semantic training complete. Model saved to checkpoints/final_model_semantic-20.pt")


Semantic training complete. Model saved to checkpoints/final_model_semantic-20.pt
